# Experiment: Batch Size × Optimizer Interaction
How does batch size interact with optimizer choice (SGD, Adam, Lion)?  
We train a CNN across a grid of batch sizes and optimizers, with/without LR warmup.


## Imports & device

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
from lion_pytorch import Lion  # pip install lion-pytorch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Dataset

In [2]:
transform = transforms.Compose([
    # Convert PIL image to PyTorch tensor and normalize pixel values to [0, 1]
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], # CIFAR-10 dataset mean
                         std=[0.2470, 0.2435, 0.2616]) # CIFAR-10 dataset std
])
train_dataset = datasets.CIFAR10(root="data", train=True,  download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root="data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:29<00:00, 5.76MB/s] 


In [7]:
n_classes = 10

## CNN Model

In [3]:
# CNN model adapted to CIFAR-10 (32x32 RGB images), 
# 3 convolutional layers and 2 fully connected layers
class CNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 32, kernel_size=3, padding="same"), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.conv2 = nn.Sequential(nn.Conv2d(32, 32, kernel_size=3, padding="same"), nn.ReLU(inplace=True), nn.AvgPool2d(2))
        self.conv3 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, padding="same"), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.fc1 = nn.Sequential(nn.Linear(4 * 4 * 64, 256), nn.ReLU())
        self.fc2 = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return self.fc2(x)

## Optimizer & Scheduler factories

In [ ]:
# Different optimizers start with their known optimal learning rates,
# all having same learning rate would be unfair comparison
BASE_LR    = {"sgd": 0.01, "adam": 1e-3, "lion": 1e-4}
BASE_BATCH = 32  # reference batch size for linear LR scaling

def get_optimizer(name, model, batch_size):
    lr = BASE_LR[name] * (batch_size / BASE_BATCH) # multiply LR by batch_size / BASE_BATCH for linear scaling
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    elif name == "adam":
        return optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    elif name == "lion":
        return Lion(model.parameters(), lr=lr, weight_decay=1e-2)

# LR warmup
def get_scheduler(optimizer, warmup, epochs):
    if not warmup:
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    return optim.lr_scheduler.SequentialLR(optimizer, schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5),
        optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - 5)
    ], milestones=[5])

## Training loop

In [5]:
def evaluate(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            total_loss += loss_fn(logits, y).item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total, total_loss / total


def train_one_run(optimizer_name, batch_size, warmup, train_dataset, val_dataset, n_classes, epochs=30):
    model   = CNN(n_classes=n_classes).to(device)
    loss_fn = nn.CrossEntropyLoss()

    train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_dl   = DataLoader(val_dataset,   batch_size=256,        shuffle=False)

    optimizer = get_optimizer(optimizer_name, model, batch_size)
    scheduler = get_scheduler(optimizer, warmup, epochs)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        correct, total, total_loss = 0, 0, 0.0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
        scheduler.step()

        train_acc, train_loss = correct / total, total_loss / total
        val_acc,   val_loss   = evaluate(model, val_dl)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"  epoch {epoch+1:02d}/{epochs} | train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

    return history

## Run all experiments

In [ ]:
OPTIMIZERS  = ["sgd", "adam", "lion"]
BATCH_SIZES = [32, 256, 1024]
WARMUPS     = [False, True]
EPOCHS      = 15

all_results = {}

for opt in OPTIMIZERS:
    for bs in BATCH_SIZES:
        for warmup in WARMUPS:
            key = f"{opt}_bs{bs}_warmup{warmup}"
            print(f"\n=== {key} ===")
            all_results[key] = train_one_run(
                optimizer_name=opt,
                batch_size=bs,
                warmup=warmup,
                train_dataset=train_dataset,
                val_dataset=test_dataset,
                n_classes=n_classes,
                epochs=EPOCHS
            )

with open("results.pkl", "wb") as f:
    pickle.dump(all_results, f)
print("\nSaved to results.pkl")


=== sgd_bs32_warmupFalse ===
  epoch 01/30 | train_acc=0.456 val_acc=0.602
  epoch 02/30 | train_acc=0.636 val_acc=0.673
  epoch 03/30 | train_acc=0.709 val_acc=0.697


KeyboardInterrupt: 